# Flipkart Gridlock Hackathon 2.0

In [ ]:
!pip install pygeohash xgboost scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

print("1. Loading datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def preprocess_data(df):
    df = df.copy()

    # --- A. SPATIAL ENGINEERING ---
    # Decode Geohash to exact Latitude and Longitude
    df['Latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else 0)
    df['Longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else 0)

    # --- B. TEMPORAL ENGINEERING ---
    # Extract continuous timeline from Day, Hour, and Minute
    df['hours'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[0]) if pd.notnull(x) and ':' in str(x) else 0)
    df['mins'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[1]) if pd.notnull(x) and ':' in str(x) else 0)
    df['time_continuous'] = 24 * 60 * (df['day'] - 1) + 60 * df['hours'] + df['mins']

    # --- C. MISSING VALUE HANDLING ---
    # 1. Numerical: Fill missing temperatures with the dataset median, lanes with mode
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    df['NumberofLanes'] = df['NumberofLanes'].fillna(df['NumberofLanes'].mode()[0])

    # 2. Categorical: Treat missing values as their own 'Unknown' category
    cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown')

    return df

print("2. Preprocessing features (Extracting time and spatial data)...")
train_prepped = preprocess_data(train)
test_prepped = preprocess_data(test)

print("3. Encoding categorical variables...")
cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in cat_cols:
    le = LabelEncoder()
    # Fit on both train and test simultaneously so no classes are missed
    le.fit(pd.concat([train_prepped[col], test_prepped[col]]))
    train_prepped[col] = le.transform(train_prepped[col])
    test_prepped[col] = le.transform(test_prepped[col])

# Define the final features list
features = [
    'time_continuous', 'Latitude', 'Longitude',
    'day', 'hours', 'mins',
    'RoadType', 'NumberofLanes', 'LargeVehicles',
    'Landmarks', 'Temperature', 'Weather'
]

X_train = train_prepped[features]
y_train = train_prepped['demand']
X_test = test_prepped[features]

print("4. Training Optimized XGBoost Model...")
# These hyperparameters are tuned for high-cardinality spatial tabular data
model = XGBRegressor(
    n_estimators=1200,       # Very high number of trees to catch every nuance
    learning_rate=0.02,      # Slow learning rate for maximum precision
    max_depth=12,            # Deep enough to see complex patterns, shallow enough to prevent overfitting
    subsample=0.85,          # Randomly sample data to boost generalization (anti-overfitting)
    colsample_bytree=0.85,   # Randomly sample features per tree
    tree_method='hist',      # Highly optimized for Colab / Large Datasets
    n_jobs=-1,               # Use all Colab CPU cores
    random_state=42
)

model.fit(X_train, y_train)

print("5. Generating Predictions...")
preds = model.predict(X_test)

# Crucial constraint: Traffic demand cannot drop below zero.
preds = np.clip(preds, 0, None)

print("6. Saving final submission...")
sub = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

filename = 'colab_master_submission.csv'
sub.to_csv(filename, index=False)
print(f"SUCCESS! Output saved to the Colab file browser as '{filename}'. Download and submit it!")